# Notebook 5 — Demo Langsung & Eksperimen 5 Studi Kasus Kegagalan

Notebook ini menjalankan dua komponen akhir tugas Evaluasi CPMK 3 (Visi Komputer ITS):
1. **Demo langsung (10 menit):** inferensi foto baru (atau dari `test_own`) pada ketiga model fraksi 100%, ubah confidence threshold secara interaktif, hitung Mask AP satu kelas tertentu, dan penjelasan baris kode kunci.
2. **5 studi kasus kegagalan (`test_own`) + eksperimen pembuktian kecil:** masing-masing dikaitkan dengan mekanisme arsitektur (one-stage CNN, two-stage CNN, transformer) dan diuji sebelum vs sesudah perlakuan.

**Pengaturan Kaggle:** Accelerator **GPU T4** (lebih cepat untuk demo; CPU juga berjalan), **Add Input**: output versi terbaru Notebook 0, 1, 2, dan 3.

In [ ]:
!pip install -q ultralytics pycocotools

## 1. Konfigurasi, Ekstraksi Data, dan Pencarian Bobot

Otomatis mencari `indoor_seg_data.zip` (Notebook 0) dan folder `runs/` (Notebook 1–3) di `/kaggle/input`.

In [ ]:
import contextlib
import glob
import io
import json
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from transformers import Mask2FormerForUniversalSegmentation
from ultralytics import YOLO

INPUT_ROOT = "/kaggle/input"
DATA_ZIP_GLOB = "/kaggle/input/**/indoor_seg_data.zip"
DATA_ROOT = Path("/tmp/data")
WORK = Path("/kaggle/working")
FIG_DIR = WORK / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "yolo11m_seg": "YOLO11m-seg",
    "mask_rcnn_r50": "Mask R-CNN R50-FPN v2",
    "mask2former_swin_t": "Mask2Former Swin-T",
}
COLORS = {"yolo11m_seg": "#2a78d6", "mask_rcnn_r50": "#eb6834", "mask2former_swin_t": "#1baf7a"}
FOCUS_TAG = "100"
IMGSZ = 640
IOU_MATCH = 0.5
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEV.type == "cuda"
print(f"Device: {DEV} (AMP={AMP})")

# Ekstrak dataset dari Notebook 0
zips = sorted(glob.glob(DATA_ZIP_GLOB, recursive=True))
assert zips, f"indoor_seg_data.zip tidak ditemukan di {DATA_ZIP_GLOB}"
if not (DATA_ROOT / "dataset_info.json").exists():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(DATA_ROOT)

INFO = json.loads((DATA_ROOT / "dataset_info.json").read_text(encoding="utf-8"))
CLASSES = INFO["classes"]
CAT_NAME = {c["coco_id"]: c["name"] for c in CLASSES}
NAME_TO_ID = {c["name"]: c["coco_id"] for c in CLASSES}
NUM_CLASSES = len(CLASSES)


def load_coco(path):
    with contextlib.redirect_stdout(io.StringIO()):
        return COCO(str(path))


SPLITS = [s for s in ["valid", "test", "test_own"] if (DATA_ROOT / "annotations" / f"{s}.json").exists()]
GT = {s: load_coco(DATA_ROOT / "annotations" / f"{s}.json") for s in SPLITS}
GT_TRAIN = load_coco(DATA_ROOT / "annotations" / "train_100.json")
TRAIN_COUNT = {cid: len(GT_TRAIN.getAnnIds(catIds=[cid])) for cid in CAT_NAME}
print(f"Dataset siap: {NUM_CLASSES} kelas, split={SPLITS}")


def find_run_dir(model_key, tag=FOCUS_TAG, need="weights"):
    cands = [Path(p) for p in glob.glob(f"{INPUT_ROOT}/**/{model_key}_{tag}", recursive=True) if Path(p).is_dir()]
    if need == "weights":
        check = {
            "yolo11m_seg": lambda d: (d / "weights" / "best.pt").exists(),
            "mask_rcnn_r50": lambda d: (d / "weights" / "best.pth").exists(),
            "mask2former_swin_t": lambda d: (d / "weights" / "best").is_dir(),
        }[model_key]
        cands = [c for c in cands if check(c)]
    else:
        cands = [c for c in cands if (c / f"preds_{need}.json").exists()]
    assert cands, f"Folder {model_key}_{tag} (need={need}) tidak ditemukan di {INPUT_ROOT}"
    return max(cands, key=lambda c: c.stat().st_mtime)


WEIGHT_DIRS = {m: find_run_dir(m, FOCUS_TAG, need="weights") for m in MODELS}
PRED_DIRS = {m: find_run_dir(m, FOCUS_TAG, need="test_own") for m in MODELS}
for m in MODELS:
    print(f"  {MODELS[m]:22s} | weights: {WEIGHT_DIRS[m]}")

## 2. Muat 3 Model (Fraksi 100%) dan Fungsi Inferensi Seragam

Ketiga model dimuat sekali ke memori/GPU dengan praproses dan pascaproses yang **identik** dengan Notebook 1–3.
Output `predict_image(...)` seragam berupa daftar dict `{mask, cat_id, label, score, box}`.

In [ ]:
def load_all_models():
    loaded = {}

    # 1. YOLO11m-seg
    loaded["yolo11m_seg"] = YOLO(str(WEIGHT_DIRS["yolo11m_seg"] / "weights" / "best.pt"))

    # 2. Mask R-CNN R50-FPN v2 (weights=None agar tidak unduh bobot pretrained; bobot penuh dari best.pth)
    rcnn = maskrcnn_resnet50_fpn_v2(
        weights=None, weights_backbone=None, min_size=IMGSZ, max_size=IMGSZ,
        box_detections_per_img=100, box_score_thresh=0.001,
    )
    rcnn.roi_heads.box_predictor = FastRCNNPredictor(
        rcnn.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES + 1
    )
    rcnn.roi_heads.mask_predictor = MaskRCNNPredictor(
        rcnn.roi_heads.mask_predictor.conv5_mask.in_channels, 256, NUM_CLASSES + 1
    )
    rcnn.load_state_dict(
        torch.load(WEIGHT_DIRS["mask_rcnn_r50"] / "weights" / "best.pth", map_location=DEV, weights_only=True)
    )
    rcnn.to(DEV).eval()
    loaded["mask_rcnn_r50"] = rcnn

    # 3. Mask2Former Swin-T
    m2f = Mask2FormerForUniversalSegmentation.from_pretrained(
        str(WEIGHT_DIRS["mask2former_swin_t"] / "weights" / "best")
    )
    m2f.to(DEV).eval()
    loaded["mask2former_swin_t"] = m2f

    print("Ketiga model berhasil dimuat.")
    return loaded


LOADED = load_all_models()


def _m2f_preprocess(pil_img, long_side):
    w, h = pil_img.size
    s = long_side / max(h, w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    x = TF.normalize(TF.to_tensor(pil_img.resize((nw, nh), Image.BILINEAR)), MEAN, STD)
    out = x.new_zeros((3, long_side, long_side))
    out[:, :nh, :nw] = x
    return out, (h, w), (nh, nw)


@torch.no_grad()
def _m2f_instance_inference(class_logits, mask_logits, valid_hw, orig_hw, padded, topk=100, chunk=10):
    # Baris kunci #3: upsample mask per potongan `chunk` query agar hemat VRAM (hindari OOM pada gambar besar)
    scores = class_logits.float().softmax(-1)[:, :-1]
    num_classes = scores.shape[1]
    top_scores, idx = scores.flatten().topk(min(topk, scores.numel()))
    q, cls = idx // num_classes, idx % num_classes
    binary = torch.empty((len(q), *orig_hw), dtype=torch.bool)
    mask_score = []
    for s in range(0, len(q), chunk):
        m = F.interpolate(mask_logits[q[s:s + chunk]][None].float(), size=(padded, padded),
                          mode="bilinear", align_corners=False)[0]
        m = m[:, :valid_hw[0], :valid_hw[1]]
        m = F.interpolate(m[None], size=orig_hw, mode="bilinear", align_corners=False)[0]
        b = m > 0
        mask_score.append((m.sigmoid() * b).flatten(1).sum(1) / (b.flatten(1).sum(1) + 1e-6))
        binary[s:s + len(b)] = b.cpu()
    return binary.numpy(), cls.cpu().numpy(), (top_scores * torch.cat(mask_score)).cpu().numpy()


@torch.no_grad()
def predict_image(model_key, pil_img, conf_thresh=0.25, imgsz=IMGSZ, yolo_iou=0.7):
    """Jalankan inferensi 1 gambar PIL dan kembalikan (dets, ms)."""
    pil_img = pil_img.convert("RGB")
    w, h = pil_img.size
    dets = []
    t0 = time.time()

    if model_key == "yolo11m_seg":
        # Baris kunci #1: retina_masks=True menghasilkan mask pada resolusi asli gambar (bukan 160x160)
        r = LOADED[model_key].predict(
            pil_img, imgsz=imgsz, conf=conf_thresh, iou=yolo_iou,
            max_det=100, retina_masks=True, device=0 if DEV.type == "cuda" else "cpu", verbose=False,
        )[0]
        if r.masks is not None and len(r.boxes) > 0:
            masks = r.masks.data.cpu().numpy() > 0.5
            if masks.shape[1:] != (h, w):
                masks = np.stack([cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
                                  for m in masks]) > 0
            xyxy = r.boxes.xyxy.cpu().numpy()
            cls = r.boxes.cls.cpu().numpy().astype(int)
            conf = r.boxes.conf.cpu().numpy()
            for m, b, c, s in zip(masks, xyxy, cls, conf):
                if m.any() and s >= conf_thresh:
                    cid = int(c) + 1
                    dets.append({"mask": m, "cat_id": cid, "label": CAT_NAME[cid], "score": float(s), "box": b})

    elif model_key == "mask_rcnn_r50":
        model = LOADED[model_key]
        # Sesuaikan ukuran input jika eksperimen mengubah imgsz
        model.transform.min_size = (imgsz,)
        model.transform.max_size = imgsz
        x = TF.to_tensor(pil_img).to(DEV)
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
            out = model([x])[0]
        masks = (out["masks"][:, 0] > 0.5).cpu().numpy()
        boxes = out["boxes"].float().cpu().numpy()
        labels = out["labels"].cpu().numpy()
        scores = out["scores"].float().cpu().numpy()
        for m, b, c, s in zip(masks, boxes, labels, scores):
            if m.any() and s >= conf_thresh:
                cid = int(c)
                dets.append({"mask": m, "cat_id": cid, "label": CAT_NAME[cid], "score": float(s), "box": b})

    elif model_key == "mask2former_swin_t":
        model = LOADED[model_key]
        x, orig_hw, valid_hw = _m2f_preprocess(pil_img, imgsz)
        pm = torch.zeros((1, imgsz, imgsz), dtype=torch.long, device=DEV)
        pm[:, :valid_hw[0], :valid_hw[1]] = 1
        with torch.autocast(DEV.type, dtype=torch.float16, enabled=AMP):
            out = model(pixel_values=x[None].to(DEV), pixel_mask=pm)
        binary, cls, scores = _m2f_instance_inference(
            out.class_queries_logits[0], out.masks_queries_logits[0], valid_hw, orig_hw, imgsz
        )
        for m, c, s in zip(binary, cls, scores):
            if m.any() and s >= conf_thresh:
                cid = int(c) + 1
                ys, xs = np.where(m)
                b = np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=float)
                dets.append({"mask": m, "cat_id": cid, "label": CAT_NAME[cid], "score": float(s), "box": b})

    ms = (time.time() - t0) * 1000.0
    dets.sort(key=lambda d: d["score"], reverse=True)
    return dets, ms


# Helper visualisasi & pencocokan GT satu gambar (konsisten dengan NB 4)
PALETTE = plt.cm.tab20(np.linspace(0, 1, 20))[:, :3]


def draw_masks(ax, pil_img, items, title=""):
    """items: list of (mask_bool, color_rgb, text_label)."""
    arr = np.array(pil_img).astype(float) / 255.0
    ax.imshow(arr)
    for mask, col, txt in items:
        if mask is None or not mask.any():
            continue
        rgb = mcolors.to_rgb(col)
        overlay = np.zeros((*mask.shape, 4))
        overlay[mask, :3] = rgb
        overlay[mask, 3] = 0.40
        ax.imshow(overlay)
        cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in cnts:
            pts = cnt[:, 0, :]
            ax.plot(pts[:, 0], pts[:, 1], color=rgb, lw=1.4)
        if txt:
            ys, xs = np.where(mask)
            ax.text(xs.mean(), ys.mean(), txt, color="white", fontsize=7.5, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=0.15", fc=rgb, ec="none", alpha=0.82))
    ax.set_title(title, fontsize=9.5)
    ax.axis("off")


def find_test_own_image(prefix):
    """Cari metadata gambar test_own berdasarkan awalan nama file, mis. 'oklusi_03'."""
    gt = GT["test_own"]
    matches = [im for im in gt.imgs.values() if im["file_name"].startswith(prefix + "_")]
    assert matches, f"Gambar dengan awalan '{prefix}_' tidak ditemukan di test_own"
    im = matches[0]
    path = DATA_ROOT / "images" / "test_own" / im["file_name"]
    anns = gt.loadAnns(gt.getAnnIds(imgIds=[im["id"]]))
    gt_items = [{"mask": gt.annToMask(a).astype(bool), "cat_id": a["category_id"],
                 "label": CAT_NAME[a["category_id"]]} for a in anns]
    return im, Image.open(path).convert("RGB"), gt_items


def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    if inter == 0:
        return 0.0
    return float(inter / np.logical_or(a, b).sum())


def eval_single_image(dets, gt_items, iou_thresh=IOU_MATCH):
    """Greedy matching (urutan skor tertinggi) untuk 1 gambar -> dict TP, FP, FN, P, R, F1."""
    used = set()
    tp, fp_cls, fp_loc, fp_bg = 0, 0, 0, 0
    matched_flags = []
    for d in dets:
        best_iou, best_j = 0.0, -1
        for j, g in enumerate(gt_items):
            if j in used or g["cat_id"] != d["cat_id"]:
                continue
            iou = mask_iou(d["mask"], g["mask"])
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh:
            used.add(best_j)
            tp += 1
            matched_flags.append(True)
        else:
            matched_flags.append(False)
            any_iou = max((mask_iou(d["mask"], g["mask"]) for g in gt_items), default=0.0)
            if any_iou >= iou_thresh:
                fp_cls += 1
            elif any_iou >= 0.1:
                fp_loc += 1
            else:
                fp_bg += 1
    fp = len(dets) - tp
    fn = len(gt_items) - tp
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"n_gt": len(gt_items), "n_pred": len(dets), "TP": tp, "FP": fp, "FN": fn,
            "salah_kelas": fp_cls, "lokalisasi": fp_loc, "latar": fp_bg,
            "P": p, "R": r, "F1": f1, "matched": matched_flags}

---
## Bagian A — Demo Langsung (10 Menit)

Bagian ini disiapkan agar saat demo cukup mengubah **1–2 variabel** lalu tekan `Shift+Enter`:
1. **Demo 1:** Inferensi foto baru (atau dari `test_own`) + ubah `CONF_THRESH`.
2. **Demo 2:** Perbandingan 3 nilai threshold sekaligus (`0.10`, `0.25`, `0.50`).
3. **Demo 3:** Hitung Mask AP **satu kelas tertentu** (`eval_ap_kelas`).
4. **Demo 4:** Ringkasan baris kode kunci yang siap dijelaskan.

### Demo 1 — Inferensi Foto Baru & Ubah Confidence Threshold

- Untuk memakai **foto baru** saat demo: unggah foto ke Kaggle (mis. `/kaggle/input/.../foto_baru.jpg` atau `/kaggle/working/foto.jpg`) lalu isi `CUSTOM_IMAGE_PATH`.
- Jika `CUSTOM_IMAGE_PATH = None`, otomatis memakai foto `DEMO_TEST_OWN_PREFIX` dari `test_own`.
- Ubah `CONF_THRESH` (mis. `0.15`, `0.25`, `0.50`) lalu jalankan ulang sel ini.

In [ ]:
CUSTOM_IMAGE_PATH = None          # isi path foto baru saat demo, atau None untuk pakai test_own
DEMO_TEST_OWN_PREFIX = "normal_02"
CONF_THRESH = 0.25                # <--- ubah nilai ini saat demo (mis. 0.10 / 0.25 / 0.50)

if CUSTOM_IMAGE_PATH is not None:
    demo_img = Image.open(CUSTOM_IMAGE_PATH).convert("RGB")
    demo_title = Path(CUSTOM_IMAGE_PATH).name
    demo_gt = None
else:
    im_meta, demo_img, demo_gt = find_test_own_image(DEMO_TEST_OWN_PREFIX)
    demo_title = f"{DEMO_TEST_OWN_PREFIX} ({im_meta['width']}x{im_meta['height']})"

ncols = 4 if demo_gt is not None else 3
fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4.5))
col_idx = 0
if demo_gt is not None:
    gt_draw = [(g["mask"], PALETTE[g["cat_id"] % 20], g["label"]) for g in demo_gt]
    draw_masks(axes[0], demo_img, gt_draw, title=f"Ground Truth ({len(demo_gt)} objek)\n{demo_title}")
    col_idx = 1

rows = []
for k, m in enumerate(MODELS):
    dets, ms = predict_image(m, demo_img, conf_thresh=CONF_THRESH)
    items = [(d["mask"], PALETTE[d["cat_id"] % 20], f"{d['label']} {d['score']:.2f}") for d in dets]
    sub = f"{MODELS[m]}\n{len(dets)} deteksi | {ms:.0f} ms | thr={CONF_THRESH}"
    if demo_gt is not None:
        ev = eval_single_image(dets, demo_gt)
        sub += f" | F1={ev['F1']:.2f}"
        rows.append({"model": MODELS[m], "deteksi": len(dets), "TP": ev["TP"], "FP": ev["FP"],
                     "FN": ev["FN"], "F1": round(ev["F1"], 3), "ms": round(ms, 1)})
    else:
        rows.append({"model": MODELS[m], "deteksi": len(dets), "ms": round(ms, 1),
                     "kelas_terdeteksi": ", ".join(f"{d['label']}({d['score']:.2f})" for d in dets[:6])})
    draw_masks(axes[col_idx + k], demo_img, items, title=sub)

plt.tight_layout()
plt.savefig(FIG_DIR / "demo_inferensi.png", dpi=150, bbox_inches="tight")
plt.show()
print(pd.DataFrame(rows).to_string(index=False))

### Demo 2 — Efek Mengubah Confidence Threshold (`0.10` vs `0.25` vs `0.50`)

Menunjukkan langsung trade-off **Precision vs Recall** saat threshold dinaikkan/diturunkan pada satu model.

In [ ]:
DEMO_MODEL = "mask2former_swin_t"   # pilih: 'yolo11m_seg', 'mask_rcnn_r50', 'mask2former_swin_t'
THRESHOLDS = [0.10, 0.25, 0.50]

fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(15, 4.5))
thr_rows = []
for ax, thr in zip(axes, THRESHOLDS):
    dets, ms = predict_image(DEMO_MODEL, demo_img, conf_thresh=thr)
    items = [(d["mask"], PALETTE[d["cat_id"] % 20], f"{d['label']} {d['score']:.2f}") for d in dets]
    title = f"{MODELS[DEMO_MODEL]} | thr={thr}\n{len(dets)} deteksi"
    row = {"threshold": thr, "deteksi": len(dets)}
    if demo_gt is not None:
        ev = eval_single_image(dets, demo_gt)
        title += f" | P={ev['P']:.2f} R={ev['R']:.2f} F1={ev['F1']:.2f}"
        row.update({"TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
                    "Precision": round(ev["P"], 3), "Recall": round(ev["R"], 3), "F1": round(ev["F1"], 3)})
    thr_rows.append(row)
    draw_masks(ax, demo_img, items, title=title)

plt.tight_layout()
plt.savefig(FIG_DIR / "demo_threshold.png", dpi=150, bbox_inches="tight")
plt.show()
print(pd.DataFrame(thr_rows).to_string(index=False))

### Demo 3 — Hitung Mask AP Satu Kelas Tertentu

Memakai prediksi tersimpan (`preds_<split>.json`) dan `COCOeval` dengan `params.catIds = [cid]`, sehingga konsisten 100% dengan evaluasi utama dan berjalan < 1 detik.

In [ ]:
def eval_ap_kelas(class_name, split="test_own"):
    assert class_name in NAME_TO_ID, f"Kelas '{class_name}' tidak ada di 74 kelas dataset."
    cid = NAME_TO_ID[class_name]
    gt = GT[split]
    n_gt = len(gt.getAnnIds(catIds=[cid]))
    n_train = TRAIN_COUNT[cid]
    print(f"Kelas: '{class_name}' (id={cid}) | split={split} | GT={n_gt} instance | train_100={n_train} instance")
    if n_gt == 0:
        print("Kelas ini tidak memiliki ground truth di split tersebut.")
        return None

    rows = []
    for m in MODELS:
        preds = json.loads((PRED_DIRS[m] / f"preds_{split}.json").read_text(encoding="utf-8"))
        # Filter prediksi hanya untuk kelas ini (atau evaluasi penuh lalu batasi catIds)
        with contextlib.redirect_stdout(io.StringIO()):
            dt = gt.loadRes(preds) if preds else COCO()
            ev = COCOeval(gt, dt, "segm")
            ev.params.catIds = [cid]
            ev.evaluate()
            ev.accumulate()
            ev.summarize()
        n_pred_cls = sum(1 for d in preds if d["category_id"] == cid and d["score"] >= 0.25)
        rows.append({
            "model": MODELS[m],
            "Mask AP (.50:.95)": round(float(ev.stats[0]), 4),
            "AP50": round(float(ev.stats[1]), 4),
            "AP75": round(float(ev.stats[2]), 4),
            "AR100": round(float(ev.stats[8]), 4),
            "pred (>=0.25)": n_pred_cls,
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


# Daftar kelas yang muncul di test_own (untuk referensi cepat saat demo)
own_cids = sorted({a["category_id"] for a in GT["test_own"].anns.values()})
print("Kelas tersedia di test_own:", ", ".join(CAT_NAME[c] for c in own_cids))
print()

# <--- Ubah nama kelas di sini saat demo (mis. 'chair', 'bottle', 'laptop', 'wall-other-merged', 'toothbrush')
_ = eval_ap_kelas("chair", split="test_own")

### Demo 4 — Catatan Baris Kode Kunci (Siap Dijelaskan Saat Demo)

1. **YOLO11m-seg (`retina_masks=True`):**
   Secara default, YOLO11-seg menghasilkan proto-mask resolusi rendah (160×160) yang di-crop/resize kasar. `retina_masks=True` melakukan proyeksi & upsampling mask langsung ke resolusi asli gambar sebelum thresholding `> 0.5`, sehingga tepi mask lebih tajam saat dievaluasi di `AP75`/`AP95`.
2. **Mask R-CNN (`MaskRCNNPredictor(..., 256, NUM_CLASSES + 1)`):**
   Mengganti kepala prediksi mask bawaan COCO (91 kelas) menjadi `74 + 1` kelas (`+1` untuk kelas latar/background indeks `0`), menggunakan fitur 256 kanal hasil `RoIAlign` 14×14 → upsample 28×28.
3. **Mask2Former (`for s in range(0, len(q), chunk)` di `_m2f_instance_inference`):**
   100 query mask jika di-upsample sekaligus ke foto beresolusi tinggi (mis. 10–12 MP) membutuhkan ~1,3 GB/MP VRAM dan memicu OOM di GPU T4. Memproses per potongan `chunk=10` query lalu menyalin mask biner ke CPU menghasilkan keluaran **identik secara matematis** dengan pemakaian VRAM 10× lebih kecil.
4. **Evaluasi seragam (`ev.params.catIds = [cid]` pada `COCOeval(..., 'segm')`):**
   Semua model dievaluasi dengan `pycocotools` yang sama (area dihitung dari poligon/RLE mask, `maxDets=100`, tanpa bias validator internal masing-masing framework).

---
## Bagian B — 5 Studi Kasus Kegagalan (`test_own`) + Eksperimen Pembuktian Kecil

Setiap studi kasus diambil dari foto kegagalan nyata di `test_own`, dikaitkan dengan mekanisme internal model, dan diuji dengan **eksperimen pembuktian kecil** (sebelum vs sesudah perlakuan) pada ketiga model fraksi 100% (`CONF_EXP = 0.25`).

In [ ]:
CONF_EXP = 0.25
CASE_SUMMARY = []


def crop_image_and_gt(pil_img, gt_items, box_norm, min_keep_frac=0.10):
    """Crop gambar & GT mask sesuai kotak ternormalisasi (x1, y1, x2, y2) dalam [0, 1]."""
    w, h = pil_img.size
    x1, y1, x2, y2 = int(box_norm[0] * w), int(box_norm[1] * h), int(box_norm[2] * w), int(box_norm[3] * h)
    cropped = pil_img.crop((x1, y1, x2, y2))
    new_gt = []
    for g in gt_items:
        orig_area = g["mask"].sum()
        cm = g["mask"][y1:y2, x1:x2]
        if orig_area > 0 and (cm.sum() / orig_area) >= min_keep_frac and cm.sum() >= 10:
            new_gt.append({"mask": cm, "cat_id": g["cat_id"], "label": g["label"]})
    return cropped, new_gt


def enhance_low_light(pil_img, gamma=0.55, clip_limit=3.0):
    """Pencerahan gambar redup: gamma correction + CLAHE pada kanal L (LAB)."""
    bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    bgr = cv2.LUT(bgr, lut)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(l)
    rgb = cv2.cvtColor(cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def suppress_glare(pil_img, gamma=1.35, clip_limit=2.5):
    """Redam silau / highlight berlebih: kompresi highlight (gamma > 1) + CLAHE lokal."""
    bgr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    bgr = cv2.LUT(bgr, lut)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8)).apply(l)
    rgb = cv2.cvtColor(cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)


def run_case_experiment(case_id, title, img_before, gt_before, img_after, gt_after,
                        before_label="Sebelum (asli)", after_label="Sesudah (eksperimen)",
                        conf_before=CONF_EXP, conf_after=CONF_EXP,
                        imgsz_before=IMGSZ, imgsz_after=IMGSZ,
                        yolo_iou_before=0.7, yolo_iou_after=0.7, save_name=None):
    """Bandingkan 3 model sebelum vs sesudah perlakuan, cetak tabel, simpan gambar 2x4."""
    fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
    rows = []

    for row_idx, (img, gt, lbl, cf, sz, yiou) in enumerate([
        (img_before, gt_before, before_label, conf_before, imgsz_before, yolo_iou_before),
        (img_after, gt_after, after_label, conf_after, imgsz_after, yolo_iou_after),
    ]):
        gt_draw = [(g["mask"], PALETTE[g["cat_id"] % 20], g["label"]) for g in gt]
        draw_masks(axes[row_idx, 0], img, gt_draw, title=f"GT ({len(gt)} objek)\n{lbl}")
        for k, m in enumerate(MODELS):
            dets, _ = predict_image(m, img, conf_thresh=cf, imgsz=sz, yolo_iou=yiou)
            ev = eval_single_image(dets, gt)
            items = [
                (d["mask"], "#2a78d6" if ok else "#eb6834", f"{d['label']} {d['score']:.2f}")
                for d, ok in zip(dets, ev["matched"])
            ]
            draw_masks(axes[row_idx, 1 + k], img, items,
                       title=f"{MODELS[m]}\nTP={ev['TP']}/{ev['n_gt']} FP={ev['FP']} F1={ev['F1']:.2f}")
            rows.append({
                "kasus": case_id, "perlakuan": lbl, "model": MODELS[m],
                "GT": ev["n_gt"], "pred": ev["n_pred"], "TP": ev["TP"], "FP": ev["FP"], "FN": ev["FN"],
                "salah_kelas": ev["salah_kelas"], "F1": round(ev["F1"], 3),
            })

    fig.suptitle(f"Studi Kasus {case_id}: {title}  (Biru=TP, Oranye=FP)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    if save_name:
        plt.savefig(FIG_DIR / f"{save_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

    df = pd.DataFrame(rows)
    piv = df.pivot(index="model", columns="perlakuan", values=["TP", "FP", "F1"])
    print(df.to_string(index=False))
    CASE_SUMMARY.extend(rows)
    return df

### Studi Kasus 1 — Oklusi Berat & Objek Saling Menumpuk (`oklusi_03`)

- **Gejala:** Foto terburuk di `test_own` (F1 = 0,000 di ketiga model). Berisi 5 `towel` saling menumpuk/tergantung berhimpitan + 2 `bottle` sangat kecil (0,6% luas).
- **Mekanisme model:**
  - **YOLO11m-seg & Mask R-CNN:** proposal/bbox berhimpitan dengan IoU tinggi saling menekan di tahap **NMS**, dan fitur kecil (<1% luas) hilang oleh downsampling.
  - **Mask2Former:** `towel` termasuk kelas langka (14 contoh di train), dan tumpukan kain bertekstur seragam menyulitkan pemisahan query.
- **Eksperimen pembuktian kecil:**
  1. **Eksperimen 1a (Threshold & NMS):** turunkan `conf` (`0.25 → 0.05`) dan longgarkan NMS IoU YOLO (`0.7 → 0.9`) untuk menguji apakah objek sebenarnya terdeteksi dengan skor rendah/tertekan NMS.
  2. **Eksperimen 1b (Crop/Zoom pada area objek):** perbesar ukuran relatif `towel` & `bottle` di frame untuk menguji batas skala fitur.

In [ ]:
im1, img1, gt1 = find_test_own_image("oklusi_03")

# Eksperimen 1a: turunkan confidence threshold + longgarkan NMS IoU YOLO
print("--- Eksperimen 1a: Turunkan threshold (0.25 -> 0.05) & NMS IoU YOLO (0.7 -> 0.9) ---")
_ = run_case_experiment(
    "1a", "Oklusi berat (oklusi_03) — Uji Threshold & NMS",
    img1, gt1, img1, gt1,
    before_label="Asli (conf=0.25, iou=0.7)", after_label="Longgar (conf=0.05, iou=0.9)",
    conf_before=0.25, conf_after=0.05, yolo_iou_before=0.7, yolo_iou_after=0.9,
    save_name="kasus_1a_oklusi03_thresh",
)

# Eksperimen 1b: crop otomatis ketat mengelilingi union seluruh GT (+margin 10%)
union1 = np.logical_or.reduce([g["mask"] for g in gt1])
ys, xs = np.where(union1)
h1, w1 = union1.shape
pad_x, pad_y = int(0.10 * w1), int(0.10 * h1)
box1 = (max(0, xs.min() - pad_x) / w1, max(0, ys.min() - pad_y) / h1,
        min(w1, xs.max() + pad_x) / w1, min(h1, ys.max() + pad_y) / h1)
img1_crop, gt1_crop = crop_image_and_gt(img1, gt1, box1)

print("\n--- Eksperimen 1b: Crop/Zoom ke area objek ---")
_ = run_case_experiment(
    "1b", "Oklusi berat (oklusi_03) — Uji Crop/Zoom",
    img1, gt1, img1_crop, gt1_crop,
    before_label="Asli (full frame)", after_label="Crop/Zoom area objek",
    save_name="kasus_1b_oklusi03_crop",
)

### Studi Kasus 2 — Kelas Tak Pernah Dilihat (*Unseen*) vs Objek Sangat Kecil (`normal_14`)

- **Gejala:** F1 rata-rata 0,074 (YOLO 0,000, Mask R-CNN 0,000, Mask2Former 0,222). Memuat 2 `toothbrush` (**0 contoh di train**), 2 `spoon` (0,2–0,3% luas), 1 `fork` (4 contoh di train, 0,3% luas), dan latar `wall-tile` (98,5% luas).
- **Mekanisme model:**
  - Ketiga model bersifat **closed-set classifier**: bobot klasifikasi kelas `toothbrush` tidak pernah menerima gradien positif selama training, sehingga mustahil memprediksi `toothbrush` secara benar.
  - Sementara `spoon` dan `fork` terlalu kecil (<0,3% luas gambar) sehingga tergerus stride backbone (stride 8–32).
- **Eksperimen pembuktian kecil (Crop/Zoom ketat pada area objek kecil):**
  Crop/zoom diarahkan khusus ke bounding box gabungan objek-objek kecil (`spoon`, `fork`, `toothbrush`). Hipotesis: **kelas *seen* (`spoon`/`fork`) berpeluang terdeteksi saat ukurannya membesar, sedangkan kelas *unseen* (`toothbrush`) tetap 0 TP** — memisahkan secara tegas kegagalan resolusi vs kegagalan data train.

In [ ]:
im2, img2, gt2 = find_test_own_image("normal_14")

# Cari bounding box objek kecil (bukan wall-tile) untuk crop/zoom
small_gt2 = [g for g in gt2 if g["label"] != "wall-tile"]
union2 = np.logical_or.reduce([g["mask"] for g in small_gt2])
ys, xs = np.where(union2)
h2, w2 = union2.shape
pad_x, pad_y = int(0.12 * w2), int(0.12 * h2)
box2 = (max(0, xs.min() - pad_x) / w2, max(0, ys.min() - pad_y) / h2,
        min(w2, xs.max() + pad_x) / w2, min(h2, ys.max() + pad_y) / h2)
img2_crop, gt2_crop = crop_image_and_gt(img2, gt2, box2)

_ = run_case_experiment(
    "2", "Kelas unseen (toothbrush) vs objek kecil (normal_14) — Crop/Zoom",
    img2, gt2, img2_crop, gt2_crop,
    before_label="Asli (full frame)", after_label="Crop/Zoom area objek kecil",
    save_name="kasus_2_normal14_unseen_vs_small",
)

# Cek khusus TP kelas seen (spoon/fork) vs unseen (toothbrush) sebelum & sesudah zoom
print("\nRincian deteksi kelas unseen (toothbrush) vs seen kecil (spoon, fork) setelah Crop/Zoom:")
for m in MODELS:
    dets_c, _ = predict_image(m, img2_crop, conf_thresh=CONF_EXP)
    seen_pred = [d["label"] for d in dets_c if d["label"] in ("spoon", "fork")]
    unseen_pred = [d["label"] for d in dets_c if d["label"] == "toothbrush"]
    all_pred = [f"{d['label']}({d['score']:.2f})" for d in dets_c]
    print(f"  {MODELS[m]:22s} | toothbrush={len(unseen_pred)} | spoon/fork={len(seen_pred)} | semua={all_pred}")

### Studi Kasus 3 — Adegan Padat Banyak Wadah Bertumpuk (`oklusi_04`)

- **Gejala:** F1 rata-rata 0,097 dengan **13 objek** (`bowl`×3, `cup`×3, `plate`×1, `spoon`×2, `wall-tile`×2, `wall-other-merged`, `floor-other-merged`).
- **Mekanisme model:**
  - Banyak objek berkelas mirip secara visual (`bowl`/`cup`/`plate`) saling menutupi pada satu rak.
  - Pada **YOLO11m-seg** (proto-mask global 160×160) dan **Mask R-CNN** (mask head 28×28 per RoI), fitur dua objek yang bertumpuk mudah bercampur di batas tepi → salah kelas dan mask bocor.
  - Pada resolusi input 640 px, objek kecil (`spoon`, `cup` di belakang) hanya menempati beberapa piksel pada feature map P3/P4.
- **Eksperimen pembuktian kecil:** Naikkan resolusi inferensi (`imgsz`: `640 → 960`) untuk menguji apakah detail spasial yang lebih rapat meningkatkan TP & F1.

In [ ]:
im3, img3, gt3 = find_test_own_image("oklusi_04")

_ = run_case_experiment(
    "3", "Adegan padat 13 objek (oklusi_04) — Uji Resolusi 640 -> 960",
    img3, gt3, img3, gt3,
    before_label="Resolusi 640", after_label="Resolusi 960",
    imgsz_before=640, imgsz_after=960,
    save_name="kasus_3_oklusi04_res960",
)

### Studi Kasus 4 — Cahaya Redup pada Objek Kecil (`redup_07` vs Pasangannya `oklusi_01`)

- **Gejala:** Pada `redup_07` (`knife`, `spoon`×2, `fork`), YOLO11m-seg dan Mask R-CNN keduanya jatuh ke **F1 = 0,000**, sedangkan Mask2Former masih mendapat **F1 = 0,400**. Pada pemandangan yang sama dalam cahaya normal (`oklusi_01`), YOLO dan Mask R-CNN naik ke F1 = 0,182.
- **Mekanisme model:**
  - Cahaya redup menurunkan kontras lokal dan ketajaman tepi, sehingga filter konvolusi lokal awal di backbone CNN (CSPDarknet/ResNet-50) gagal menghasilkan aktivasi kuat untuk objek tipis/kecil.
  - Self-attention global pada Swin-T/Mask2Former lebih mampu memanfaatkan konteks bentuk keseluruhan meski tepi lokal lemah.
- **Eksperimen pembuktian kecil:** Terapkan **pencerahan gamma (0,55) + CLAHE** pada `redup_07`, lalu bandingkan metrik **sebelum vs sesudah pencerahan** dan bandingkan dengan pasangan cahaya normal **`oklusi_01`**.

In [ ]:
im4, img4, gt4 = find_test_own_image("redup_07")
img4_bright = enhance_low_light(img4, gamma=0.55, clip_limit=3.0)

_ = run_case_experiment(
    "4", "Cahaya redup (redup_07) — Uji Pencerahan Gamma + CLAHE",
    img4, gt4, img4_bright, gt4,
    before_label="Asli (redup_07)", after_label="Dicerahkan (gamma+CLAHE)",
    save_name="kasus_4_redup07_brighten",
)

# Pembanding: pasangan cahaya normal (oklusi_01) dari pemandangan yang sama
_, img4_pair, gt4_pair = find_test_own_image("oklusi_01")
print("\nPembanding langsung — pasangan cahaya normal (oklusi_01):")
pair_rows = []
for m in MODELS:
    dets_p, _ = predict_image(m, img4_pair, conf_thresh=CONF_EXP)
    ev_p = eval_single_image(dets_p, gt4_pair)
    pair_rows.append({"foto": "oklusi_01 (normal)", "model": MODELS[m],
                      "TP": ev_p["TP"], "FP": ev_p["FP"], "FN": ev_p["FN"], "F1": round(ev_p["F1"], 3)})
print(pd.DataFrame(pair_rows).to_string(index=False))

### Studi Kasus 5 — Silau / Pantulan Cahaya Kuat (`silau_01` vs Pasangannya `normal_10`)

- **Gejala:** Pada `silau_01` (7 objek: `door-stuff`, `laptop`, `light`, `mouse*`, `plug`, `table-merged`, `wall-other-merged`), **YOLO11m-seg** paling terdampak (AP 0,123, F1 0,267) dibanding **Mask R-CNN** (AP 0,386, F1 0,462) dan **Mask2Former** (F1 0,600).
- **Mekanisme model:**
  - Piksel jenuh (*clipped highlights*) dari pantulan lampu menghapus tekstur permukaan meja/laptop tepat di titik tengah grid prediksi; detektor one-stage anchor-free (YOLO) yang mengandalkan fitur lokal sekitar pusat objek mengalami penurunan skor confidence atau salah lokalisasi.
  - Two-stage RoIAlign (Mask R-CNN) dan global self-attention (Mask2Former) mengagregasi area lebih luas sehingga lebih tahan terhadap hotspot silau lokal.
- **Eksperimen pembuktian kecil:** Terapkan **kompresi highlight (gamma 1,35) + CLAHE lokal** untuk meredam silau pada `silau_01`, lalu bandingkan sebelum vs sesudah serta bandingkan dengan pasangan **`normal_10`**.

In [ ]:
im5, img5, gt5 = find_test_own_image("silau_01")
img5_fixed = suppress_glare(img5, gamma=1.35, clip_limit=2.5)

_ = run_case_experiment(
    "5", "Silau / pantulan kuat (silau_01) — Uji Kompresi Highlight + CLAHE",
    img5, gt5, img5_fixed, gt5,
    before_label="Asli (silau_01)", after_label="Redam silau (gamma+CLAHE)",
    save_name="kasus_5_silau01_deglare",
)

# Pembanding: pasangan cahaya normal (normal_10)
_, img5_pair, gt5_pair = find_test_own_image("normal_10")
print("\nPembanding langsung — pasangan cahaya normal (normal_10):")
pair5_rows = []
for m in MODELS:
    dets_p, _ = predict_image(m, img5_pair, conf_thresh=CONF_EXP)
    ev_p = eval_single_image(dets_p, gt5_pair)
    pair5_rows.append({"foto": "normal_10 (normal)", "model": MODELS[m],
                       "TP": ev_p["TP"], "FP": ev_p["FP"], "FN": ev_p["FN"], "F1": round(ev_p["F1"], 3)})
print(pd.DataFrame(pair5_rows).to_string(index=False))

## Ringkasan Eksperimen & Daftar Output

In [ ]:
summary_df = pd.DataFrame(CASE_SUMMARY)
summary_df.to_csv(WORK / "ringkasan_eksperimen_kasus.csv", index=False)
print("=== RINGKASAN SEMUA EKSPERIMEN STUDI KASUS ===")
print(summary_df.to_string(index=False))

print("\nFile tersimpan di /kaggle/working:")
for p in sorted(WORK.glob("*.csv")):
    print("  ", p.name)
print("Gambar di /kaggle/working/figures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print("  ", p.name)